# 6 自定义量化 A8W8 matmul 算子开发并接入 Qwen3-8B

本章是整个启航营的**核心实践环节**，将学习到的所有知识沉淀转化为一次完整的自定义算子开发实战。

实践的环境准备、提交方式可参考[启航营课程实践说明](https://gitcode.com/cann/cann-launch-camp/blob/master/2026/University/HIT/First-session/README.md)。

### 本章定位

本章将开发一个自定义的量化 matmul 算子 `QmmCustom`，通过 A8W8 量化（Activation INT8 + Weight INT8）减少计算量和内存占用，从而提升推理性能。这是本次启航营中综合性最强的一章——涵盖了从算子概念理解、Tiling 设计、Kernel 实现到算子编译、接入网络与功能性能测试的全链路。

### 通过实践达成的学习目标

- **理解量化推理原理**：掌握 A8W8 量化方案（INT8 激活 + INT8 权重）的数学原理与 Scale 反量化机制，理解 perChannelScale 与 perTokenScale 的作用。
- **设计 Tiling 数据结构与函数**：根据算子原型定义自行设计 TilingData 结构体，实现多核分块的 Tiling 函数，理解 Cube 计算单元的分块策略。
- **实现 Cube+Vector 双路径 Kernel**：完成 Cube-only（INT32 输出）和 Cube+Vector 协作（反量化 BF16 输出）两条 Kernel 路径的实现，理解昇腾 AI Core 的 Cube 与 Vector 协作编程模型。
- **完成算子编译与 Torch 接入**：编译自定义算子并通过 Torch 接口接入，打通“算子开发 → 框架调用”的链路。
- **测试算子功能与性能，尝试优化迭代**：验证自定义算子的精度正确性，测试单算子与网络性能，并根据测试结果对 Tiling 或 Kernel 进行优化迭代。

### 本章大纲

- 环境准备
- A8W8 量化 matmul 算子概念
- 自定义算子实现
    - **任务一：自行设计tilingdata结构体并实现tiling函数**
    - **任务二：实现A8W8输入，INT32输出的kernel**
    - **任务三：实现A8W8输入，经过perChannelScale和perTokenScale反量化输出BF16的kernel**
- 编译自定义算子
    - **任务四：成功编译自定义算子**
- 测试自定义算子功能性能
    - **任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程**
- 自定义算子接入模型
- 模型性能测试
    - **任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程**
- 提交要求
---

## 1 环境准备

定位教程目录、准备运行目录，并把 CANN 环境变量导入当前 Notebook 进程：

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

print('[环境准备] 开始', flush=True)

def locate_repo_root():
    repo_roots = []
    try:
        cwd = Path.cwd()
        lineage = [cwd, *cwd.parents]
        repo_roots.extend(lineage)
        repo_roots.extend(base / 'cann-learning-hub' for base in lineage)
        for base in lineage:
            try:
                repo_roots.extend(base.glob('*/cann-learning-hub'))
            except OSError:
                pass
    except FileNotFoundError:
        pass
    repo_roots.append(Path('/opt/atomgit/cann-learning-hub'))

    seen = set()
    for repo_root in repo_roots:
        key = str(repo_root)
        if key in seen:
            continue
        seen.add(key)
        if (repo_root / 'tutorials/llm_inference/qwen3_8b/src').exists():
            return repo_root
    raise FileNotFoundError('Cannot locate cann-learning-hub repository root.')

REPO_ROOT = locate_repo_root()
os.chdir(REPO_ROOT)
TUTORIAL_DIR = REPO_ROOT / 'tutorials' / 'llm_inference' / 'qwen3_8b'
TUTORIAL_SRC_DIR = TUTORIAL_DIR / 'src'
if str(TUTORIAL_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIAL_SRC_DIR))
SRC_DIR = TUTORIAL_DIR / 'src' / 'op_custom'

if not os.environ.get('ASCEND_TOOLKIT_HOME'):
    raise EnvironmentError('ASCEND_TOOLKIT_HOME is not set. Please activate the CANN environment.')
print('CANN包路径 =', os.environ.get('ASCEND_TOOLKIT_HOME'))
cann_script = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'set_env.sh'
env_cmd = f'source {cann_script} && env'
env = subprocess.check_output(f"bash -lc '{env_cmd}'", shell=True, text=True, cwd=REPO_ROOT)
for line in env.splitlines():
    if '=' in line:
        os.environ.__setitem__(*line.split('=', 1))

RECIPE_ROOT = TUTORIAL_DIR / 'src/inference_scripts/recipe_qwen3_8b'
QMM_CUSTOM_DIR = SRC_DIR / 'qmm_custom'

print('教程目录 =', TUTORIAL_DIR)
print('推理模型目录 =', RECIPE_ROOT)
print('算子相关文件目录 =', SRC_DIR)
print('算子实现目录 =', QMM_CUSTOM_DIR)
print('[环境准备] 完成', flush=True)

[环境准备] 开始
CANN包路径 = /home/developer/Ascend/cann-8.5.2
教程目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b
推理模型目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b
算子相关文件目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom
算子实现目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom
[环境准备] 完成


安装所需的依赖：

In [2]:
%pip install -r {SRC_DIR}/requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple --trusted-host pypi.tuna.tsinghua.edu.cn

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.


---
## 2 A8W8 量化 matmul 算子概念

### 2.1 什么是 A8W8 量化

A8W8 量化是 LLM 推理中常用的一种量化方案：
- **A（Activation）**：将输入激活值从 BF16/FP16 量化为 **INT8**
- **W（Weight）**：将权重从 BF16/FP16 量化为 **INT8**
- **反量化**：计算结果 `INT8 × INT8 → INT32` 后，通过 Scale 反量化回浮点精度

### 2.2 算子规格

在上一章中，通过分析量化模型的profiling，归纳出了自定义算子的输入输出规格：

| 算子名称 | QmmCustom |
|------|-----|
| 输入 x1 (A) | INT8, ND 格式, shape [M, K] |
| 输入 x2 (B) | INT8, **FRACTAL_NZ** 格式, shape [K, N] |
| 输入 scale | FLOAT32, ND 格式, shape [N] |
| 输入 pertoken_scale | FLOAT32, ND 格式, shape [M]（可选） |
| 输出 y | 有 perTokenScale → BF16，无 → INT32；ND 格式, shape [M, N] |
| 架构 | ascend910b |

**注意**：x2 权重矩阵使用 **FRACTAL_NZ** 格式，这是昇腾 AI Core Cube 单元的高效存储格式，能提升 matmul 计算性能。

### 2.3 Scale 模式

QmmCustom 算子支持两种 Scale：

| Scale 类型 | Shape | 作用 | 
|-----------|-------|------|
| **perChannelScale** | `[N]` | 对每个输出通道单独缩放 |
| **perTokenScale** | `[M]` | 对每行输入单独缩放 |

计算公式：
```
result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale
```

## 3 自定义算子实现

与前面学习过的内容一致，本次实践采用 `<<<>>>` Kernel 直调方式开发上述原型的量化 matmul 算子。直调方式通过 `kernel<<<numBlocks, dynUBufSize, stream>>>(args)` 语法在 Host 侧直接启动 Kernel，适合快速验证算子功能与性能。

本节包含如下三个任务：

### 任务一：自行设计 TilingData 结构体并实现 Tiling 函数

根据算子原型与 Kernel 的实现需求，自行设计 `QmmCustomTilingData` 结构体，并实现 `CalcQmmTiling` 函数。Tiling 函数需根据输入的 M、N、K 维度，结合平台硬件资源（L1/L0A/L0B/UB 内存限制、可用核数），计算多核分块参数。可借助 `platform_ascendc::PlatformAscendCManager` 获取硬件信息。

> **Tiling 设计提示**：
> 
> 设计 TilingData 时需要考虑以下要素：
> 1. **`TCubeTiling`**：CANN Matmul 库提供的标准 Tiling 结构体，包含 M/N/K 的分块大小（`baseM`、`baseN`、`baseK`）、使用的核数（`usedCoreNum`）等字段。可通过 `GetMatmulTiling` API 自动计算。
> 2. **`isPertoken`**：标记是否走反量化路径，用于 Kernel 内部分支选择。
> 3. **`workspaceSize`**：Path 2 中 Cube 写入 INT32 中间结果所需的 GM workspace 大小。
> 4. **多核分块策略**：将 M 维度或 N 维度分配到多个核上并行计算，每个核处理一个分块。分块大小需平衡 L1/L0A/L0B 缓存利用率与多核负载均衡。

### 任务二：实现 A8W8 输入、INT32 输出的 Kernel

实现 `QmmCubeBasicKernel` 类，完成纯 Cube 路径的 INT8 矩阵乘法计算。

计算公式： ```result_int32 = x1_int8 @ x2_int8```

### 任务三：实现 A8W8 输入、经 perChannelScale 和 perTokenScale 反量化输出 BF16 的 Kernel

实现 `QmmPertokenKernel` 类，在 Cube 计算的基础上增加 Vector 反量化路径。

计算公式： ```result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale```

其中 `perChannelScale` 沿 N 维广播，`perTokenScale` 沿 M 维广播，两者均为 FLOAT32 类型。

---

### 执行路径说明

在 Host 侧的 `qmm_custom` 函数中，根据是否传入 `pertoken_scale` 设置 `isPertoken` 标志，Kernel 启动时据此选择执行路径：

| 路径 | 条件 | 计算流程 | 输出类型 |
|------|------|----------|----------|
| **Path 1: Cube-only** | `isPertoken == 0` | INT8 matmul → INT32 | INT32 |
| **Path 2: Cube+Vector** | `isPertoken == 1` | INT8 matmul → INT32 → dequant(scale×pertoken) → BF16 | BF16 |

**Path 1（Cube-only）**：仅使用 AI Core 的 Cube 计算单元，完成 INT8×INT8 矩阵乘法后直接将 INT32 结果写入输出 GM，无 Vector 参与。对应 `QmmCubeBasicKernel` 的实现。

**Path 2（Cube+Vector）**：Cube 单元先完成 INT8 matmul 得到 INT32 中间结果并写入 GM workspace；随后 Vector 单元从 GM 读取 INT32 结果，依次完成 `Cast`（INT32→FLOAT32）、乘以 `perChannelScale` 和 `perTokenScale`、`Cast`（FLOAT32→BF16）等反量化操作，最终将 BF16 结果写入输出 GM。对应 `QmmPertokenKernel` 的实现。

Kernel 入口函数 `qmm_custom_kernel` 使用 `__mix__(1, 2)` 属性声明，表示为每个 Block 分配 1 个 Cube 核和 2 个 Vector 核，使 Cube 与 Vector 可在同一 Kernel 中协同工作。Kernel 内部通过 `tilingData.isPertoken` 判断走哪条路径，两条路径复用相同的 Tiling 结构和 Kernel 启动入口。

Tiling、Kernel、Torch 接口的实现均位于 `qmm_custom.asc` 文件中。文件已给出整体框架与函数签名，请自行完成下方代码中的 TODO 部分，以实现完整的自定义算子：

In [3]:
%%writefile {QMM_CUSTOM_DIR}/qmm_custom.asc
#include "acl/acl_base.h"
#include "kernel_operator.h"

// CANN 8.5.x 的自定义 C/V 混合模式要求在包含 Matmul 接口前定义该宏。
#define ASCENDC_CUBE_ONLY
#include "lib/matmul_intf.h"

#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"
#include "torch_npu/csrc/core/npu/NPUStream.h"
#include <torch/extension.h>

#include <algorithm>
#include <cmath>
#include <cstdint>
#include <limits>

using namespace AscendC;

namespace {

// Cube 的 M 方向以 16 个元素为基本粒度。
constexpr uint32_t kCubeMAlign = 16;
// INT8 FRACTAL_NZ 的 B 矩阵内轴（逻辑 N）C0 为 32。
// 每个 N 分块及其起始偏移必须按 32 对齐。
constexpr uint32_t kNzNAlign = 32;
constexpr uint32_t kNzKAlign = 32;
constexpr uint32_t kWorkspaceAlign = 512;
constexpr uint32_t kVecTileElems = 1024;
constexpr uint16_t kCubeToVectorFlag = 0x8;

inline uint32_t CeilDivU32(uint32_t x, uint32_t y) {
  return (x + y - 1U) / y;
}

inline uint32_t AlignUpU32(uint32_t x, uint32_t align) {
  return CeilDivU32(x, align) * align;
}

inline uint64_t AlignUpU64(uint64_t x, uint64_t align) {
  return ((x + align - 1U) / align) * align;
}

struct GridPlan {
  uint32_t singleM;
  uint32_t singleN;
  uint32_t mBlocks;
  uint32_t nBlocks;
  uint32_t usedCores;
};

// 选择逻辑核网格：
// 1) decode/small-M 场景只切 N，避免每核 M 过小；
// 2) prefill/large-M 场景二维切分，并优先使用更多 AIC；
// 3) M 块按 16 对齐；N 块按 32 对齐。INT8 FRACTAL_NZ 的
//    逻辑 N 是内轴，C0_size=32，不能在 16 列边界切分。
static GridPlan ChooseGrid(uint32_t M, uint32_t N, uint32_t maxCores) {
  GridPlan best{AlignUpU32(M, kCubeMAlign), AlignUpU32(N, kNzNAlign),
                1U, 1U, 1U};
  uint32_t bestUsed = 0;
  double bestPenalty = std::numeric_limits<double>::infinity();

  const uint32_t maxMParts = (M <= 64U) ? 1U : maxCores;
  for (uint32_t requestedMParts = 1; requestedMParts <= maxMParts;
       ++requestedMParts) {
    const uint32_t singleM =
        AlignUpU32(CeilDivU32(M, requestedMParts), kCubeMAlign);
    const uint32_t actualMParts = CeilDivU32(M, singleM);

    for (uint32_t requestedNParts = 1; requestedNParts <= maxCores;
         ++requestedNParts) {
      const uint32_t singleN =
          AlignUpU32(CeilDivU32(N, requestedNParts), kNzNAlign);
      const uint32_t actualNParts = CeilDivU32(N, singleN);
      const uint32_t used = actualMParts * actualNParts;
      if (used == 0U || used > maxCores) {
        continue;
      }

      // Cube 的常用高效基本块接近 128x256，故倾向 singleN ≈ 2*singleM。
      const double aspect =
          static_cast<double>(singleN) /
          (2.0 * static_cast<double>(std::max(singleM, 1U)));
      const double aspectPenalty = std::fabs(std::log2(std::max(aspect, 1e-9)));
      // 同核数时，优先较小的单核输出块，降低尾块和中间结果延迟。
      const double areaPenalty =
          static_cast<double>(singleM) * static_cast<double>(singleN) * 1e-12;
      const double penalty = aspectPenalty + areaPenalty;

      if (used > bestUsed || (used == bestUsed && penalty < bestPenalty)) {
        bestUsed = used;
        bestPenalty = penalty;
        best = {singleM, singleN, actualMParts, actualNParts, used};
      }
    }
  }
  return best;
}

inline int32_t SelectBaseM(uint32_t singleM) {
  if (singleM >= 128U) return 128;
  if (singleM >= 64U) return 64;
  if (singleM >= 32U) return 32;
  return 16;
}

inline int32_t SelectBaseN(uint32_t singleN) {
  if (singleN >= 256U) return 256;
  if (singleN >= 128U) return 128;
  if (singleN >= 64U) return 64;
  if (singleN >= 32U) return 32;
  return 16;
}

} // namespace

#pragma pack(push, 8)
struct alignas(8) QmmCustomTilingData {
  TCubeTiling cubeTilingData;
  uint32_t isPertoken;
  uint32_t M;
  uint32_t N;
  uint32_t K;
  uint32_t singleCoreM;
  uint32_t singleCoreN;
  uint32_t mBlocks;
  uint32_t nBlocks;
  uint32_t vecTileElems;
  uint32_t reserved;
  uint64_t userWorkspaceOffset;
  uint64_t userWorkspaceSize;
  uint64_t workspaceSize;
};
#pragma pack(pop)

// Host 侧 Tiling。适配 CANN 8.5.2 / dav-2201（910C）。
static QmmCustomTilingData CalcQmmTiling(uint32_t isPertoken, uint32_t M,
                                         uint32_t N, uint32_t K) {
  QmmCustomTilingData tilingData{};
  auto ascendcPlatform =
      platform_ascendc::PlatformAscendCManager::GetInstance();
  TORCH_CHECK(ascendcPlatform != nullptr,
              "QmmCustom: failed to get AscendC platform information.");

  const uint32_t aicNum = ascendcPlatform->GetCoreNumAic();
  TORCH_CHECK(aicNum > 0U, "QmmCustom: no available AIC core.");

  const GridPlan plan = ChooseGrid(M, N, aicNum);

  matmul_tiling::MultiCoreMatmulTiling tilingApi(*ascendcPlatform);
  TORCH_CHECK(tilingApi.SetDim(plan.usedCores) != -1,
              "QmmCustom: SetDim failed.");
  TORCH_CHECK(
      tilingApi.SetAType(matmul_tiling::TPosition::GM,
                         matmul_tiling::CubeFormat::ND,
                         matmul_tiling::DataType::DT_INT8, false) != -1,
      "QmmCustom: SetAType failed.");
  TORCH_CHECK(
      tilingApi.SetBType(matmul_tiling::TPosition::GM,
                         matmul_tiling::CubeFormat::NZ,
                         matmul_tiling::DataType::DT_INT8, false) != -1,
      "QmmCustom: SetBType failed.");
  TORCH_CHECK(
      tilingApi.SetCType(matmul_tiling::TPosition::GM,
                         matmul_tiling::CubeFormat::ND,
                         matmul_tiling::DataType::DT_INT32) != -1,
      "QmmCustom: SetCType failed.");
  TORCH_CHECK(
      tilingApi.SetBiasType(matmul_tiling::TPosition::GM,
                            matmul_tiling::CubeFormat::ND,
                            matmul_tiling::DataType::DT_INT32) != -1,
      "QmmCustom: SetBiasType failed.");

  TORCH_CHECK(tilingApi.SetOrgShape(M, N, K) != -1,
              "QmmCustom: SetOrgShape failed.");
  TORCH_CHECK(tilingApi.SetShape(M, N, K) != -1,
              "QmmCustom: SetShape failed.");
  TORCH_CHECK(
      tilingApi.SetSingleShape(plan.singleM, plan.singleN, K) != -1,
      "QmmCustom: SetSingleShape failed.");
  TORCH_CHECK(tilingApi.EnableBias(false) != -1,
              "QmmCustom: EnableBias failed.");

  // 128x256 是官方 Matmul 调优案例中的高吞吐基本块；小 M/N 自动降级。
  const int32_t baseM = SelectBaseM(plan.singleM);
  const int32_t baseN = SelectBaseN(plan.singleN);
  TORCH_CHECK(tilingApi.SetFixSplit(baseM, baseN, -1) != -1,
              "QmmCustom: SetFixSplit failed.");
  TORCH_CHECK(tilingApi.SetBufferSpace(-1, -1, -1) != -1,
              "QmmCustom: SetBufferSpace failed.");

  const int64_t ret = tilingApi.GetTiling(tilingData.cubeTilingData);
  TORCH_CHECK(ret != -1, "QmmCustom: Matmul tiling failed for M=", M,
              ", N=", N, ", K=", K, ".");

  // 必须以 GetTiling 返回的 singleCoreM/N 为最终依据。Tiling API
  // 可能为了满足 L1/L0 约束微调请求值；若仍使用请求值计算 GM 偏移，
  // 多核之间会读取/写入错误的矩阵分块。
  const uint32_t actualSingleM =
      static_cast<uint32_t>(tilingData.cubeTilingData.singleCoreM);
  const uint32_t actualSingleN =
      static_cast<uint32_t>(tilingData.cubeTilingData.singleCoreN);
  TORCH_CHECK(actualSingleM > 0U && actualSingleN > 0U,
              "QmmCustom: invalid single-core shape returned by tiling.");
  TORCH_CHECK((actualSingleN % kNzNAlign) == 0U,
              "QmmCustom: singleCoreN must be 32-aligned for INT8 "
              "FRACTAL_NZ, actual=", actualSingleN);

  const uint32_t actualMBlocks = CeilDivU32(M, actualSingleM);
  const uint32_t actualNBlocks = CeilDivU32(N, actualSingleN);
  const uint32_t activeCores = actualMBlocks * actualNBlocks;
  TORCH_CHECK(activeCores <= tilingData.cubeTilingData.usedCoreNum,
              "QmmCustom: Matmul usedCoreNum is smaller than required. "
              "required=", activeCores, ", actual=",
              tilingData.cubeTilingData.usedCoreNum);

  const uint64_t systemWorkspaceSize =
      static_cast<uint64_t>(ascendcPlatform->GetLibApiWorkSpaceSize());
  const uint64_t userWorkspaceOffset =
      AlignUpU64(systemWorkspaceSize, kWorkspaceAlign);
  const uint64_t userWorkspaceSize =
      isPertoken
          ? AlignUpU64(static_cast<uint64_t>(M) * N * sizeof(int32_t),
                       kWorkspaceAlign)
          : 0U;

  tilingData.isPertoken = isPertoken;
  tilingData.M = M;
  tilingData.N = N;
  tilingData.K = K;
  tilingData.singleCoreM = actualSingleM;
  tilingData.singleCoreN = actualSingleN;
  tilingData.mBlocks = actualMBlocks;
  tilingData.nBlocks = actualNBlocks;
  tilingData.vecTileElems = kVecTileElems;
  tilingData.userWorkspaceOffset = userWorkspaceOffset;
  tilingData.userWorkspaceSize = userWorkspaceSize;
  tilingData.workspaceSize = userWorkspaceOffset + userWorkspaceSize;
  return tilingData;
}

using QmmAType =
    matmul::MatmulType<TPosition::GM, CubeFormat::ND, int8_t, false>;
using QmmBType =
    matmul::MatmulType<TPosition::GM, CubeFormat::NZ, int8_t, false>;
using QmmCType =
    matmul::MatmulType<TPosition::GM, CubeFormat::ND, int32_t, false>;
using QmmBiasType =
    matmul::MatmulType<TPosition::GM, CubeFormat::ND, int32_t, false>;
// MDL 提高 GM->L1 大包搬运与流水效率；CANN 8.5.2 支持 INT8 Matmul。
using QmmMatmul =
    matmul::Matmul<QmmAType, QmmBType, QmmCType, QmmBiasType,
                   CFG_MDL>;

struct QmmCoreSlice {
  uint32_t logicCoreId;
  uint32_t mIndex;
  uint32_t nIndex;
  uint32_t mStart;
  uint32_t nStart;
  uint32_t tailM;
  uint32_t tailN;
  uint64_t aOffset;
  uint64_t bOffset;
  uint64_t cOffset;
};

__aicore__ inline QmmCoreSlice MakeCoreSlice(
    const QmmCustomTilingData *__restrict tilingData) {
  uint32_t logicCoreId = 0;
  if ASCEND_IS_AIC {
    logicCoreId = GetBlockIdx();
  } else {
    // __mix__(1,2): 每个逻辑 AIC 对应两个 AIV。
    logicCoreId = GetBlockIdx() / 2U;
  }

  QmmCoreSlice slice{};
  slice.logicCoreId = logicCoreId;
  slice.mIndex = logicCoreId % tilingData->mBlocks;
  slice.nIndex = logicCoreId / tilingData->mBlocks;
  slice.mStart = slice.mIndex * tilingData->singleCoreM;
  slice.nStart = slice.nIndex * tilingData->singleCoreN;
  slice.tailM =
      (slice.mStart < tilingData->M)
          ? min(tilingData->singleCoreM, tilingData->M - slice.mStart)
          : 0U;
  slice.tailN =
      (slice.nStart < tilingData->N)
          ? min(tilingData->singleCoreN, tilingData->N - slice.nStart)
          : 0U;
  slice.aOffset = static_cast<uint64_t>(slice.mStart) * tilingData->K;
  // B 为逻辑 [K,N] 的 FRACTAL_NZ。N 方向偏移 nStart 对应物理偏移 nStart*K。
  slice.bOffset = static_cast<uint64_t>(slice.nStart) * tilingData->K;
  slice.cOffset = static_cast<uint64_t>(slice.mStart) * tilingData->N +
                  slice.nStart;
  return slice;
}

// 任务二：A8W8 -> INT32，只在 AIC 上运行。
class QmmCubeBasicKernel {
public:
  __aicore__ inline QmmCubeBasicKernel() {}

  __aicore__ inline void Init(
      GM_ADDR x1, GM_ADDR x2, GM_ADDR y,
      const QmmCustomTilingData *__restrict tilingData) {
    tilingData_ = tilingData;
    slice_ = MakeCoreSlice(tilingData);
    if ASCEND_IS_AIC {
      x1Gm_.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x1),
                            tilingData->M * tilingData->K);
      x2Gm_.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x2),
                            tilingData->K * tilingData->N);
      yGm_.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t *>(y),
                           tilingData->M * tilingData->N);
      cubeTiling_ = tilingData->cubeTilingData;
    }
  }

  __aicore__ inline void ProcessCube(QmmMatmul &mm) {
    if (slice_.tailM == 0U || slice_.tailN == 0U) {
      return;
    }
    // 显式设置完整原始 shape，使 ND 输出行跨度和 NZ 输入地址解释
    // 始终以全局 M/N/K 为准。
    mm.SetOrgShape(static_cast<int>(tilingData_->M),
                   static_cast<int>(tilingData_->N),
                   static_cast<int>(tilingData_->K));
    mm.SetTensorA(x1Gm_[slice_.aOffset], false);
    mm.SetTensorB(x2Gm_[slice_.bOffset], false);
    mm.SetTail(static_cast<int>(slice_.tailM),
               static_cast<int>(slice_.tailN),
               static_cast<int>(tilingData_->K));
    mm.IterateAll(yGm_[slice_.cOffset]);
    mm.End();
  }

  TCubeTiling cubeTiling_;

private:
  const QmmCustomTilingData *tilingData_{nullptr};
  QmmCoreSlice slice_{};
  GlobalTensor<int8_t> x1Gm_;
  GlobalTensor<int8_t> x2Gm_;
  GlobalTensor<int32_t> yGm_;
};

// 任务三：A8W8 -> INT32(Cube) -> BF16(Vector)。
// AIC 先写用户 workspace；同一逻辑核的两个 AIV 分摊 M 行做反量化。
class QmmPertokenKernel {
public:
  __aicore__ inline QmmPertokenKernel() {}

  __aicore__ inline void Init(
      GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR pertokenScale, GM_ADDR y,
      GM_ADDR workspace, const QmmCustomTilingData *__restrict tilingData,
      TPipe *pipe) {
    tilingData_ = tilingData;
    slice_ = MakeCoreSlice(tilingData);

    if ASCEND_IS_AIC {
      x1Gm_.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x1),
                            tilingData->M * tilingData->K);
      x2Gm_.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x2),
                            tilingData->K * tilingData->N);
      cubeTiling_ = tilingData->cubeTilingData;
    }

    if ASCEND_IS_AIV {
      GM_ADDR userWorkspace = workspace + tilingData->userWorkspaceOffset;
      workspaceGm_.SetGlobalBuffer(
          reinterpret_cast<__gm__ int32_t *>(userWorkspace),
          tilingData->M * tilingData->N);
      scaleGm_.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(scale),
                               tilingData->N);
      pertokenScaleGm_.SetGlobalBuffer(
          reinterpret_cast<__gm__ float *>(pertokenScale), tilingData->M);
      yGm_.SetGlobalBuffer(reinterpret_cast<__gm__ bfloat16_t *>(y),
                           tilingData->M * tilingData->N);

      pipe->InitBuffer(mmInQueue_, 1,
                       tilingData->vecTileElems * sizeof(int32_t));
      pipe->InitBuffer(scaleQueue_, 1,
                       tilingData->vecTileElems * sizeof(float));
      pipe->InitBuffer(outQueue_, 1,
                       tilingData->vecTileElems * sizeof(bfloat16_t));
      pipe->InitBuffer(calcBuf_,
                       tilingData->vecTileElems * sizeof(float));
    }
  }

  __aicore__ inline void ProcessCube(QmmMatmul &mm, GM_ADDR workspace) {
    if (slice_.tailM == 0U || slice_.tailN == 0U) {
      CrossCoreSetFlag<0x2, PIPE_FIX>(kCubeToVectorFlag);
      return;
    }

    GM_ADDR userWorkspace = workspace + tilingData_->userWorkspaceOffset;
    GlobalTensor<int32_t> workspaceGm;
    workspaceGm.SetGlobalBuffer(
        reinterpret_cast<__gm__ int32_t *>(userWorkspace),
        tilingData_->M * tilingData_->N);

    // 显式设置完整原始 shape，使 ND 输出行跨度和 NZ 输入地址解释
    // 始终以全局 M/N/K 为准。
    mm.SetOrgShape(static_cast<int>(tilingData_->M),
                   static_cast<int>(tilingData_->N),
                   static_cast<int>(tilingData_->K));
    mm.SetTensorA(x1Gm_[slice_.aOffset], false);
    mm.SetTensorB(x2Gm_[slice_.bOffset], false);
    mm.SetTail(static_cast<int>(slice_.tailM),
               static_cast<int>(slice_.tailN),
               static_cast<int>(tilingData_->K));
    mm.IterateAll(workspaceGm[slice_.cOffset]);
    mm.End();

    // PIPE_FIX 保证 Matmul 的 GM 搬出完成后再唤醒两个 AIV 子核。
    CrossCoreSetFlag<0x2, PIPE_FIX>(kCubeToVectorFlag);
  }

  __aicore__ inline void ProcessVector() {
    // mode 2 下，一个 AIC 的 set 会通知同组两个 AIV。
    CrossCoreWaitFlag(kCubeToVectorFlag);

    if (slice_.tailM == 0U || slice_.tailN == 0U) {
      return;
    }

    const uint32_t subBlock = GetSubBlockIdx() & 1U;
    const uint32_t firstRows = (slice_.tailM + 1U) / 2U;
    const uint32_t rowOffset = (subBlock == 0U) ? 0U : firstRows;
    const uint32_t rowCount =
        (subBlock == 0U) ? firstRows : (slice_.tailM - firstRows);

    for (uint32_t localRow = 0; localRow < rowCount; ++localRow) {
      const uint32_t globalRow = slice_.mStart + rowOffset + localRow;
      const float tokenScale = pertokenScaleGm_.GetValue(globalRow);

      for (uint32_t col = 0; col < slice_.tailN;
           col += tilingData_->vecTileElems) {
        const uint32_t curN =
            min(tilingData_->vecTileElems, slice_.tailN - col);
        const uint64_t matrixOffset =
            static_cast<uint64_t>(globalRow) * tilingData_->N +
            slice_.nStart + col;
        const uint64_t scaleOffset = slice_.nStart + col;

        LocalTensor<int32_t> mmLocal =
            mmInQueue_.AllocTensor<int32_t>();
        LocalTensor<float> scaleLocal =
            scaleQueue_.AllocTensor<float>();
        DataCopy(mmLocal, workspaceGm_[matrixOffset], curN);
        DataCopy(scaleLocal, scaleGm_[scaleOffset], curN);
        mmInQueue_.EnQue(mmLocal);
        scaleQueue_.EnQue(scaleLocal);

        mmLocal = mmInQueue_.DeQue<int32_t>();
        scaleLocal = scaleQueue_.DeQue<float>();
        LocalTensor<float> calcLocal = calcBuf_.Get<float>();
        LocalTensor<bfloat16_t> outLocal =
            outQueue_.AllocTensor<bfloat16_t>();

        Cast(calcLocal, mmLocal, RoundMode::CAST_NONE, curN);
        Mul(calcLocal, calcLocal, scaleLocal, curN);
        Muls(calcLocal, calcLocal, tokenScale, curN);
        Cast(outLocal, calcLocal, RoundMode::CAST_RINT, curN);

        outQueue_.EnQue(outLocal);
        outLocal = outQueue_.DeQue<bfloat16_t>();
        DataCopy(yGm_[matrixOffset], outLocal, curN);

        outQueue_.FreeTensor(outLocal);
        scaleQueue_.FreeTensor(scaleLocal);
        mmInQueue_.FreeTensor(mmLocal);
      }
    }
  }

  TCubeTiling cubeTiling_;

private:
  const QmmCustomTilingData *tilingData_{nullptr};
  QmmCoreSlice slice_{};
  GlobalTensor<int8_t> x1Gm_;
  GlobalTensor<int8_t> x2Gm_;
  GlobalTensor<int32_t> workspaceGm_;
  GlobalTensor<float> scaleGm_;
  GlobalTensor<float> pertokenScaleGm_;
  GlobalTensor<bfloat16_t> yGm_;
  TQue<QuePosition::VECIN, 1> mmInQueue_;
  TQue<QuePosition::VECIN, 1> scaleQueue_;
  TQue<QuePosition::VECOUT, 1> outQueue_;
  TBuf<QuePosition::VECCALC> calcBuf_;
};

__global__ __mix__(1, 2) __aicore__
void qmm_custom_kernel(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale,
                       GM_ADDR pertoken_scale, GM_ADDR y, GM_ADDR workspace,
                       QmmCustomTilingData tilingData) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_MIX_AIC_1_2);
  if (workspace == nullptr) {
    return;
  }

  TPipe pipe;
  if ASCEND_IS_AIC {
    // Kernel 直调且未依赖 HAVE_WORKSPACE 时，显式设置 Matmul 系统 workspace。
    SetSysWorkspace(workspace);
    if (GetSysWorkSpacePtr() == nullptr) {
      return;
    }
  }

  if (tilingData.isPertoken == 0U) {
    QmmCubeBasicKernel kernel;
    kernel.Init(x1, x2, y, &tilingData);
    if ASCEND_IS_AIC {
      QmmMatmul mm;
      REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), mm,
                        &kernel.cubeTiling_);
      kernel.ProcessCube(mm);
    }
  } else {
    QmmPertokenKernel kernel;
    kernel.Init(x1, x2, scale, pertoken_scale, y, workspace, &tilingData,
                &pipe);
    if ASCEND_IS_AIC {
      QmmMatmul mm;
      REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), mm,
                        &kernel.cubeTiling_);
      kernel.ProcessCube(mm, workspace);
    }
    if ASCEND_IS_AIV {
      kernel.ProcessVector();
    }
  }
  pipe.Destroy();
}

namespace ascendc_ops {

at::Tensor qmm_custom(const at::Tensor &x1, const at::Tensor &x2,
                      const at::Tensor &scale,
                      const c10::optional<at::Tensor> &pertoken_scale) {
  TORCH_CHECK(x1.device().type() == c10::DeviceType::PrivateUse1,
              "QmmCustom: x1 must be an NPU tensor.");
  TORCH_CHECK(x2.device().type() == c10::DeviceType::PrivateUse1,
              "QmmCustom: x2 must be an NPU tensor.");
  TORCH_CHECK(scale.device().type() == c10::DeviceType::PrivateUse1,
              "QmmCustom: scale must be an NPU tensor.");
  TORCH_CHECK(x1.dim() == 2 && x2.dim() == 2,
              "QmmCustom: x1 and x2 must be 2-D logical tensors.");
  TORCH_CHECK(x1.scalar_type() == at::ScalarType::Char &&
                  x2.scalar_type() == at::ScalarType::Char,
              "QmmCustom: x1/x2 dtype must be int8.");
  TORCH_CHECK(scale.scalar_type() == at::ScalarType::Float,
              "QmmCustom: scale dtype must be float32.");

  const int64_t M64 = x1.size(0);
  const int64_t K64 = x1.size(1);
  const int64_t N64 = x2.size(1);
  TORCH_CHECK(x2.size(0) == K64,
              "QmmCustom: x1.size(1) must equal x2.size(0).");
  TORCH_CHECK(M64 > 0 && N64 > 0 && K64 > 0,
              "QmmCustom: M/N/K must be positive.");
  TORCH_CHECK(M64 <= std::numeric_limits<uint32_t>::max() &&
                  N64 <= std::numeric_limits<uint32_t>::max() &&
                  K64 <= std::numeric_limits<uint32_t>::max(),
              "QmmCustom: M/N/K exceeds uint32 range.");
  TORCH_CHECK((K64 % kNzKAlign) == 0,
              "QmmCustom: K must be a multiple of 32 for INT8 FRACTAL_NZ.");
  TORCH_CHECK((N64 % kNzNAlign) == 0,
              "QmmCustom: N must be a multiple of 32 for INT8 "
              "FRACTAL_NZ.");
  TORCH_CHECK(scale.numel() == N64,
              "QmmCustom: scale.numel() must equal N.");

  const uint32_t isPertoken =
      pertoken_scale.has_value() && pertoken_scale.value().defined() ? 1U : 0U;
  if (isPertoken) {
    const at::Tensor &tokenScale = pertoken_scale.value();
    TORCH_CHECK(tokenScale.device().type() == c10::DeviceType::PrivateUse1,
                "QmmCustom: pertoken_scale must be an NPU tensor.");
    TORCH_CHECK(tokenScale.scalar_type() == at::ScalarType::Float,
                "QmmCustom: pertoken_scale dtype must be float32.");
    TORCH_CHECK(tokenScale.numel() == M64,
                "QmmCustom: pertoken_scale.numel() must equal M.");
  }

  const uint32_t M = static_cast<uint32_t>(M64);
  const uint32_t N = static_cast<uint32_t>(N64);
  const uint32_t K = static_cast<uint32_t>(K64);
  auto aclStream = c10_npu::getCurrentNPUStream().stream(true);

  const at::ScalarType outputDtype =
      isPertoken ? at::ScalarType::BFloat16 : at::ScalarType::Int;
  at::Tensor y = at::empty({M64, N64}, x1.options().dtype(outputDtype));

  QmmCustomTilingData tilingData = CalcQmmTiling(isPertoken, M, N, K);
  const uint32_t numBlocks = tilingData.cubeTilingData.usedCoreNum;
  TORCH_CHECK(numBlocks > 0U, "QmmCustom: invalid blockDim.");

  at::Tensor workspaceTensor =
      at::empty({static_cast<int64_t>(tilingData.workspaceSize)},
                x1.options().dtype(at::ScalarType::Byte));

  uint8_t *pertokenPtr =
      isPertoken
          ? reinterpret_cast<uint8_t *>(
                pertoken_scale.value().mutable_data_ptr())
          : nullptr;

  // Direct launch: CANN 8.5.2 has no aclprof*Ex metadata API.
  // msprof/PyTorch Profiler can collect the kernel without a host wrapper.
  qmm_custom_kernel<<<numBlocks, 0, aclStream>>>(
      reinterpret_cast<uint8_t *>(x1.mutable_data_ptr()),
      reinterpret_cast<uint8_t *>(x2.mutable_data_ptr()),
      reinterpret_cast<uint8_t *>(scale.mutable_data_ptr()), pertokenPtr,
      reinterpret_cast<uint8_t *>(y.mutable_data_ptr()),
      workspaceTensor.data_ptr<uint8_t>(), tilingData);
  return y;
}

} // namespace ascendc_ops

TORCH_LIBRARY(ascendc_ops, m) {
  m.def("qmm_custom(Tensor x1, Tensor x2, Tensor scale, Tensor? "
        "pertoken_scale=None) -> Tensor");
}

TORCH_LIBRARY_IMPL(ascendc_ops, PrivateUse1, m) {
  m.impl("qmm_custom", TORCH_FN(ascendc_ops::qmm_custom));
}


Overwriting /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/qmm_custom.asc


---
## 4 编译自定义算子

### 任务四：成功编译自定义算子


在算子目录下执行编译命令：

In [4]:
!cd /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom && \
rm -rf build && \
mkdir build && \
cd build && \
cmake .. && \
make -j

-- System processer: aarch64
-- CMAKE_ASC_COMPILER: /home/developer/Ascend/cann-8.5.2/bin/bisheng
-- ASCEND_CANN_PACKAGE_LINUX_PATH: /home/developer/Ascend/cann-8.5.2/aarch64-linux
-- CMAKE_ASC_LLD_LINKER: /home/developer/Ascend/cann-8.5.2/aarch64-linux/ccec_compiler/bin/ld.lld
-- The CXX compiler identification is GNU 9.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Python3: /opt/buildtools/Python-3.11.4/bin/python3 (found version "3.11.4") found components: Interpreter Development Development.Module Development.Embed 
CMake Warning at /home/developer/.local/lib/python3.11/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:22 (message):
  static library kineto_LIBRARY-NOTFOUND not found.
Call Stack (most recent call first):
  /home/developer/.local/lib/python3.11/site-packages/torch/share/cmake/Torch/TorchC

编译成功后会在build目录下生成ascendc_ops.so

---
## 5 测试自定义算子功能性能

### 任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程

使用测试脚本验证 QmmCustom 算子在不同规格输入下的精度和性能。测试脚本覆盖多组 (M, K, N) 规格，分别验证 INT32 输出（无 perTokenScale）和 BF16 输出（有 perTokenScale）两种路径。

核心接口调用如下：

```python
output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu, pertoken_scale_npu)
```

### 5.1 测试算子功能

需要测试的 cases 的 shape 如下，对应的是 Qwen3-8B 模型中实际出现的调用，**全部通过**视为算子功能OK：
| M    | K     | N     |
| ---- | ----- | ----- |
| 1    | 4096  | 4096  |
| 1    | 4096  | 6144  |
| 1    | 4096  | 24576 |
| 1    | 12288 | 4096  |
| 50   | 4096  | 4096  |
| 50   | 4096  | 6144  |
| 50   | 4096  | 24576 |
| 50   | 12288 | 4096  |
| 4096 | 4096  | 4096  |
| 4096 | 4096  | 6144  |
| 4096 | 4096  | 24576 |
| 4096 | 12288 | 4096  |

In [5]:
from pathlib import Path

import torch
import torch_npu

QMM_CUSTOM_DIR = Path(
    "/mnt/workspace/gitCode/cann/cann-learning-hub/"
    "tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom"
)

LIB_PATH = QMM_CUSTOM_DIR / "build/libascendc_ops.so"

if not LIB_PATH.exists():
    raise FileNotFoundError(
        f"没有找到算子动态库：\n{LIB_PATH}\n"
        "请先确认终端中的 cmake 和 make 已成功执行。"
    )

torch.ops.load_library(str(LIB_PATH))

torch.npu.config.allow_internal_format = True
torch.npu.config.allow_hf32 = False

print(f"算子动态库加载成功：{LIB_PATH}")
import torch_npu

torch.ops.load_library(str(QMM_CUSTOM_DIR / "build/libascendc_ops.so"))

torch.npu.config.allow_internal_format = True
torch.npu.config.allow_hf32 = False
torch.npu.set_compile_mode(jit_compile=False)

# ============================================================
# 测试配置: 多组 (M, K, N) 规格
# ============================================================
TEST_SHAPES = [
    (1, 4096, 4096),
    (1, 4096, 6144),
    (1, 4096, 24576),
    (1, 12288, 4096),
    (50, 4096, 4096),
    (50, 4096, 6144),
    (50, 4096, 24576),
    (50, 12288, 4096),
    (4096, 4096, 4096),
    (4096, 4096, 6144),
    (4096, 4096, 24576),
    (4096, 12288, 4096),
]


def check_close(output, ref, dtype):
    """统一的精度比对函数，根据输出 dtype 使用不同的 tolerance"""
    if dtype == torch.int32:
        # 整数运算应精确匹配，tolerance 为 0
        rtol, atol = 0, 0
    else:
        # BF16 允许浮点舍入误差
        rtol, atol = 0.01, 0.01

    output_f = output.to(torch.float32)
    ref_f = ref.to(torch.float32)
    passed = torch.allclose(output_f, ref_f, rtol=rtol, atol=atol)

    label = 'INT32' if dtype == torch.int32 else 'BF16'
    print(f"  {label} allclose验证: {'PASS' if passed else 'FAIL'}")

    if not passed:
        print(f"  output[0,0:5] = {output[0, 0:5].tolist()}")
        print(f"  ref[0,0:5]    = {ref[0, 0:5].tolist()}")

    return passed


def test_int32(M, K, N):
    """无 pertoken_scale, 纯INT8 matmul, INT32 输出"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)  # 29: FORMAT_FRACTAL_NZ
    scale_npu = scale_cpu.npu()

    output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu, pertoken_scale=None)
    output_cpu = output_npu.cpu()

    # float64 matmul 走 BLAS 优化, 比 int32 朴素乘加快很多;
    # float64 精度足够精确表示 int32 范围内的整数结果
    ref_output = torch.matmul(x1_cpu.to(torch.float64), x2_cpu.to(torch.float64)).round().to(torch.int32)

    assert output_cpu.shape == ref_output.shape, f"shape不一致: {output_cpu.shape} vs {ref_output.shape}"
    assert output_cpu.dtype == torch.int32, f"输出dtype不正确: {output_cpu.dtype}"

    return check_close(output_cpu, ref_output, torch.int32)


def test_bf16(M, K, N):
    """有 pertoken_scale, INT8 matmul + dequant, BF16 输出"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)
    pertoken_scale_cpu = torch.randn([M], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)  # 29: FORMAT_FRACTAL_NZ
    scale_npu = scale_cpu.npu()
    pertoken_scale_npu = pertoken_scale_cpu.npu()

    output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu,
                                                  pertoken_scale=pertoken_scale_npu)
    output_cpu = output_npu.cpu()

    int32_result = torch.matmul(x1_cpu.to(torch.float64), x2_cpu.to(torch.float64)).round().to(torch.int32)
    ref_bf16 = (int32_result.to(torch.float32) * scale_cpu.unsqueeze(0) * pertoken_scale_cpu.unsqueeze(1)).to(torch.bfloat16)

    assert output_cpu.shape == ref_bf16.shape, f"shape不一致: {output_cpu.shape} vs {ref_bf16.shape}"
    assert output_cpu.dtype == torch.bfloat16, f"输出dtype不正确: {output_cpu.dtype}"

    return check_close(output_cpu, ref_bf16, torch.bfloat16)


# ============================================================
# 主循环: 对每组 (M, K, N) 执行 INT32 和 BF16 测试
# ============================================================
int32_results = []
bf16_results = []

for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
    print("=" * 60)
    print(f"规格{i}: M={M}, K={K}, N={N}")
    print("=" * 60)

    print("[INT32 测试]")
    int32_pass = test_int32(M, K, N)
    int32_results.append((M, K, N, int32_pass))

    print("[BF16 测试]")
    bf16_pass = test_bf16(M, K, N)
    bf16_results.append((M, K, N, bf16_pass))

# ============================================================
# 测试汇总
# ============================================================
print()
print("=" * 60)
print("测试汇总")
print("=" * 60)
print(f"{'规格':>24} | {'INT32':^8} | {'BF16':^8}")
print("-" * 46)
for (M, K, N, p1), (_, _, _, p2) in zip(int32_results, bf16_results):
    tag = f"M={M},K={K},N={N}"
    print(f"{tag:>24} | {'PASS' if p1 else 'FAIL':^8} | {'PASS' if p2 else 'FAIL':^8}")

int32_total = sum(p for _, _, _, p in int32_results)
bf16_total = sum(p for _, _, _, p in bf16_results)
total = len(TEST_SHAPES)
print(f"INT32: {int32_total}/{total} 通过, BF16: {bf16_total}/{total} 通过")
print("=" * 60)


/home/developer/.local/lib/python3.11/site-packages/torch_npu/__init__.py:320: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \


算子动态库加载成功：/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/build/libascendc_ops.so
规格1: M=1, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格2: M=1, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格3: M=1, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格4: M=1, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格5: M=50, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格6: M=50, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格7: M=50, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格8: M=50, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格9: M=4096, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: P

### 5.2 测试算子性能
基于torch_npu.profiler来测试算子性能，执行下面的code cell测试单算子性能数据：

In [6]:
import os
import csv
import time
import shutil
from pathlib import Path

import torch
import torch_npu

_required_names = (
    "TEST_SHAPES",
    "test_int32",
    "test_bf16",
)

_missing_names = [
    name for name in _required_names
    if name not in globals()
]

if _missing_names:
    raise RuntimeError(
        "运行 5.2 前，请先运行 5.1 中定义测试规格和测试函数的单元格。\n"
        f"当前缺少：{', '.join(_missing_names)}"
    )


# ============================================================
# Profiler 输出目录
#
# 原始代码直接使用 SRC_DIR，Restart Kernel 后如果没有重新运行
# 环境准备单元格，就会出现：
#   NameError: name 'SRC_DIR' is not defined
#
# 下面按以下顺序确定 op_custom 目录：
#   1. 已定义的 SRC_DIR
#   2. 已定义的 QMM_CUSTOM_DIR 的父目录
#   3. 教程在当前环境中的默认绝对路径
# ============================================================

if "SRC_DIR" in globals():
    OP_CUSTOM_DIR = Path(SRC_DIR)
elif "QMM_CUSTOM_DIR" in globals():
    OP_CUSTOM_DIR = Path(QMM_CUSTOM_DIR).parent
else:
    OP_CUSTOM_DIR = Path(
        "/mnt/workspace/gitCode/cann/cann-learning-hub/"
        "tutorials/llm_inference/qwen3_8b/src/op_custom"
    )

if not OP_CUSTOM_DIR.exists():
    raise FileNotFoundError(
        "没有找到 op_custom 目录：\n"
        f"{OP_CUSTOM_DIR}\n"
        "请先运行 Notebook 最前面的环境准备单元格，"
        "或者将 OP_CUSTOM_DIR 改成当前工程的实际路径。"
    )

PROF_OUTPUT_DIR = str(OP_CUSTOM_DIR / "prof_output")

# 保持原始版本默认值：不执行额外 warmup。
WARMUP_ITERS = 0


def synchronize_npu():
    """等待当前 NPU Stream 上的任务执行完成。"""
    try:
        torch.npu.synchronize()
    except Exception:
        try:
            torch_npu.npu.synchronize()
        except Exception:
            pass


def get_first_nonempty(row, keys):
    """读取 CSV 行中第一个存在且非空的字段。"""
    for key in keys:
        value = row.get(key)
        if value not in (None, ""):
            return str(value).strip()
    return ""


def read_duration_us(row):
    """
    兼容不同 CANN 8.5.2 Profiler CSV 的耗时列名，
    统一返回微秒。
    """
    for key in (
        "Duration(us)",
        "Duration (us)",
        "duration(us)",
        "Duration",
        "duration",
    ):
        value = row.get(key)

        if value not in (None, ""):
            try:
                return float(
                    str(value).replace(",", "").strip()
                )
            except ValueError:
                pass

    for key in (
        "Duration(ns)",
        "Duration (ns)",
        "duration(ns)",
    ):
        value = row.get(key)

        if value not in (None, ""):
            try:
                return (
                    float(str(value).replace(",", "").strip())
                    / 1000.0
                )
            except ValueError:
                pass

    return None


def is_qmm_custom_row(row):
    """判断 CSV 中的一行是否属于 QmmCustom。"""
    name = get_first_nonempty(
        row,
        (
            "Name",
            "Op Name",
            "OP Name",
            "Kernel Name",
            "Task Name",
            "Op Type",
            "OP Type",
        ),
    )

    normalized_name = (
        name.lower()
        .replace("_", "")
        .replace("-", "")
        .replace(" ", "")
    )

    return "qmmcustom" in normalized_name


def parse_kernel_details(csv_path, test_shapes):
    """
    解析 kernel_details.csv。

    原始版本依赖 Input Shapes 和 Output Data Types。
    在 CANN 8.5.2 中，这两个字段可能为空，因此会导致：
        IndexError: list index out of range

    修正版不再读取 Input Shapes，而是按照原始测试代码的固定执行顺序：
        shape1 INT32
        shape1 BF16
        shape2 INT32
        shape2 BF16
        ...
    将 QmmCustom 记录与 TEST_SHAPES 对齐。
    """
    durations = []
    fieldnames = []

    with open(
        csv_path,
        newline="",
        encoding="utf-8-sig",
        errors="replace",
    ) as file:
        reader = csv.DictReader(file)
        fieldnames = reader.fieldnames or []

        for row in reader:
            if not is_qmm_custom_row(row):
                continue

            duration_us = read_duration_us(row)

            if duration_us is not None:
                durations.append(duration_us)

    print(f"CSV 字段：{fieldnames}")
    print(f"找到 QmmCustom 原始记录数：{len(durations)}")

    expected = []

    for M, K, N in test_shapes:
        expected.append((M, K, N, "INT32"))
        expected.append((M, K, N, "BF16"))

    expected_count = len(expected)
    actual_count = len(durations)

    if actual_count < expected_count:
        raise RuntimeError(
            f"Profiler 中只找到 {actual_count} 条 QmmCustom 记录，"
            f"但本轮测试应至少产生 {expected_count} 条。\n"
            "请检查：\n"
            "1. 5.1 测试函数是否完整执行；\n"
            "2. Profiler 是否成功停止并导出；\n"
            "3. kernel_details.csv 中的 Kernel 名称是否包含 QmmCustom。"
        )

    # __mix__(1, 2) 内核在部分 Profiler 版本中可能将一次调用
    # 展开为多条 AIC/AIV 记录。
    if actual_count > expected_count:
        if actual_count % expected_count == 0:
            records_per_call = actual_count // expected_count

            print(
                f"检测到每次算子调用对应 {records_per_call} 条记录，"
                "按调用顺序分组，并使用组内最大耗时。"
            )

            merged = []

            for index in range(expected_count):
                begin = index * records_per_call
                end = begin + records_per_call
                merged.append(max(durations[begin:end]))

            durations = merged
        else:
            print(
                f"警告：记录数 {actual_count} 与预期数 "
                f"{expected_count} 不是整数倍。"
            )
            print(
                f"将使用最后 {expected_count} 条记录进行汇总。"
            )

            durations = durations[-expected_count:]

    rows = []

    for (M, K, N, dtype_tag), duration_us in zip(
        expected,
        durations,
    ):
        rows.append(
            {
                "M": M,
                "K": K,
                "N": N,
                "dtype": dtype_tag,
                "duration_us": float(duration_us),
            }
        )

    return rows


def find_kernel_details_csv(prof_dir, timeout_seconds=30):
    """
    等待 Profiler 完成数据导出，并递归查找最新的
    kernel_details.csv。
    """
    prof_path = Path(prof_dir)
    deadline = time.time() + timeout_seconds

    while time.time() < deadline:
        if prof_path.exists():
            candidates = list(
                prof_path.rglob("kernel_details.csv")
            )

            if candidates:
                newest = max(
                    candidates,
                    key=lambda path: path.stat().st_mtime,
                )
                return str(newest)

        time.sleep(0.5)

    return None


# ============================================================
# 主流程：清理旧数据，先 warmup，再开启 profiler
# ============================================================

# 防止上一次运行留下的 CSV 干扰本轮规格顺序。
shutil.rmtree(PROF_OUTPUT_DIR, ignore_errors=True)
os.makedirs(PROF_OUTPUT_DIR, exist_ok=True)

print(f"Profiler 输出目录：{PROF_OUTPUT_DIR}")

int32_results = []
bf16_results = []


# warmup：不采集数据，让 NPU 状态稳定
if WARMUP_ITERS > 0:
    print("=" * 60)
    print(
        f"Warmup：运行 {WARMUP_ITERS} 轮，"
        "不采集性能数据"
    )
    print("=" * 60)

    for warmup_index in range(WARMUP_ITERS):
        print(
            f"Warmup round "
            f"{warmup_index + 1}/{WARMUP_ITERS}"
        )

        for M, K, N in TEST_SHAPES:
            test_int32(M, K, N)
            test_bf16(M, K, N)

    synchronize_npu()


# ============================================================
# 创建 Profiler
# ============================================================

profiler = torch_npu.profiler.profile(
    activities=[
        torch_npu.profiler.ProfilerActivity.NPU,
        torch_npu.profiler.ProfilerActivity.CPU,
    ],
    experimental_config=torch_npu.profiler._ExperimentalConfig(
        profiler_level=(
            torch_npu.profiler.ProfilerLevel.Level1
        ),
        aic_metrics=(
            torch_npu.profiler.AiCMetrics.PipeUtilization
        ),
    ),
    on_trace_ready=(
        torch_npu.profiler.tensorboard_trace_handler(
            PROF_OUTPUT_DIR
        )
    ),
)


# ============================================================
# 开启 Profiler 采集
# ============================================================

profiler.start()

try:
    for index, (M, K, N) in enumerate(
        TEST_SHAPES,
        start=1,
    ):
        print("=" * 60)
        print(f"规格{index}: M={M}, K={K}, N={N}")
        print("=" * 60)

        print("[INT32 测试]")

        int32_pass = test_int32(M, K, N)
        synchronize_npu()

        int32_results.append(
            (M, K, N, bool(int32_pass))
        )

        profiler.step()

        print("[BF16 测试]")

        bf16_pass = test_bf16(M, K, N)
        synchronize_npu()

        bf16_results.append(
            (M, K, N, bool(bf16_pass))
        )

        profiler.step()

finally:
    synchronize_npu()
    profiler.stop()


# ============================================================
# 测试汇总
# ============================================================

print()
print("=" * 60)
print("测试汇总")
print("=" * 60)

print(
    f"{'规格':>24} | "
    f"{'INT32':^8} | "
    f"{'BF16':^8}"
)
print("-" * 46)

for (M, K, N, int32_pass), (
    _,
    _,
    _,
    bf16_pass,
) in zip(int32_results, bf16_results):
    tag = f"M={M},K={K},N={N}"

    print(
        f"{tag:>24} | "
        f"{'PASS' if int32_pass else 'FAIL':^8} | "
        f"{'PASS' if bf16_pass else 'FAIL':^8}"
    )

int32_total = sum(
    int(passed)
    for _, _, _, passed in int32_results
)

bf16_total = sum(
    int(passed)
    for _, _, _, passed in bf16_results
)

total = len(TEST_SHAPES)

print(
    f"INT32: {int32_total}/{total} 通过, "
    f"BF16: {bf16_total}/{total} 通过"
)


# ============================================================
# Profiler 数据汇总
# ============================================================

print()
print("=" * 70)
print("Profiler 数据汇总 (QmmCustom Duration)")
print("=" * 70)

csv_path = find_kernel_details_csv(
    PROF_OUTPUT_DIR,
    timeout_seconds=30,
)

print(f"原始数据路径：{csv_path}")

if csv_path:
    kernel_rows = parse_kernel_details(
        csv_path,
        TEST_SHAPES,
    )

    if kernel_rows:
        shape_map = {}

        for kernel_row in kernel_rows:
            key = (
                kernel_row["M"],
                kernel_row["K"],
                kernel_row["N"],
            )

            shape_map.setdefault(
                key,
                {},
            )[kernel_row["dtype"]] = (
                kernel_row["duration_us"]
            )

        print(
            f"{'规格 (M,K,N)':>24} | "
            f"{'INT32 Duration(us)':>18} | "
            f"{'BF16 Duration(us)':>18}"
        )
        print("-" * 70)

        for M, K, N in TEST_SHAPES:
            key = (M, K, N)

            int32_duration = shape_map.get(
                key,
                {},
            ).get(
                "INT32",
                "-",
            )

            bf16_duration = shape_map.get(
                key,
                {},
            ).get(
                "BF16",
                "-",
            )

            int32_text = (
                f"{int32_duration:.3f}"
                if isinstance(int32_duration, float)
                else str(int32_duration)
            )

            bf16_text = (
                f"{bf16_duration:.3f}"
                if isinstance(bf16_duration, float)
                else str(bf16_duration)
            )

            tag = f"M={M},K={K},N={N}"

            print(
                f"{tag:>24} | "
                f"{int32_text:>18} | "
                f"{bf16_text:>18}"
            )
    else:
        print(
            f"在 {csv_path} 中未找到 "
            "QmmCustom 算子数据"
        )
else:
    print(
        f"未在 {PROF_OUTPUT_DIR} 中找到 "
        "kernel_details.csv 文件"
    )
    print("请检查 Profiler 输出目录")

print("=" * 70)


Profiler 输出目录：/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/prof_output
规格1: M=1, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格2: M=1, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格3: M=1, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格4: M=1, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格5: M=50, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格6: M=50, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格7: M=50, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格8: M=50, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格9: M=4096, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格10: M=4096, K

---
## 6 自定义算子接入模型

### 6.1 定位替换点

Qwen3-8B 推理框架中的 W8A8 量化 matmul 通过 `CompressedTensorsW8A8Int8LinearMethod` 类实现，其 `apply_weights` 方法中使用 `torch_npu.npu_quant_matmul` 执行 INT8 量化矩阵乘法。我们需要将这个调用替换为自定义的 `torch.ops.ascendc_ops.qmm_custom` 算子。

查看原始实现：

In [7]:
!grep -n 'npu_quant_matmul' {RECIPE_ROOT}/module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py

94:        # --- 替换核心: npu_quant_matmul → qmm_custom ---


### 6.2 替换实现

在 `CompressedTensorsW8A8Int8LinearMethod` 中，将 `torch_npu.npu_quant_matmul` 替换为 `torch.ops.ascendc_ops.qmm_custom`。

替换后的方法为：

```python
torch.ops.load_library(str(QMM_CUSTOM_DIR / "build/libascendc_ops.so"))

def apply_weights(self, layer: torch.nn.Module,
                x: torch.Tensor,
                bias: Optional[torch.Tensor],
                dynamic_scale: Optional = None,
                out_dtype: Optional = torch.bfloat16
                ) -> Union[torch.Tensor, Dict[str, Any]]:
    if dynamic_scale is not None:
        x_scale = dynamic_scale
    else:
        x, x_scale = torch_npu.npu_dynamic_quant(x, smooth_scales=layer.smooth_scales)
    out_shape_dim_0 = x.size()[:-1]
    x = x.view(-1, x.size(-1))
    x_scale = x_scale.view(-1)

    # --- 替换核心: npu_quant_matmul → qmm_custom ---
    x = torch.ops.ascendc_ops.qmm_custom(x, layer.weight, 
                                layer.weight_scale.view(-1),
                                pertoken_scale=None if out_dtype == torch.int32 else x_scale)
    # --- 替换结束 ---

    x = x.view(out_shape_dim_0 + (-1, ))
    if out_dtype == torch.int32:
        return x, x_scale
    else:
        return x
```

执行如下操作替换推理网络中调用的接口实现：

In [8]:
# 6.2 替换实现: 将 torch_npu.npu_quant_matmul 替换为 torch.ops.ascendc_ops.qmm_custom

# 1: 导入libascendc_ops.so
from pathlib import Path

# Qwen3-8B 工程根目录
PROJECT_ROOT = Path(
    "/mnt/workspace/gitCode/cann/cann-learning-hub/"
    "tutorials/llm_inference/qwen3_8b"
)

# 自定义算子目录
QMM_CUSTOM_DIR = (
    PROJECT_ROOT
    / "src/op_custom/qmm_custom"
)

# 需要替换的 compressed_tensors_w8a8_int8.py
W8A8_FILE = (
    PROJECT_ROOT
    / "src/inference_scripts/recipe_qwen3_8b/module/"
      "quantization/compressed_tensors/"
      "compressed_tensors_w8a8_int8.py"
)

print("[替换] 目标文件：", W8A8_FILE)

if not W8A8_FILE.exists():
    raise FileNotFoundError(
        "没有找到目标文件：\n"
        f"{W8A8_FILE}\n\n"
        "请检查工程目录是否正确。"
    )

LIB_PATH = QMM_CUSTOM_DIR / "build/libascendc_ops.so"

if not LIB_PATH.exists():
    raise FileNotFoundError(
        "没有找到自定义算子动态库：\n"
        f"{LIB_PATH}\n\n"
        "请先完成 qmm_custom 的编译。"
    )

# 读取 Python 源文件
ct_code = W8A8_FILE.read_text(encoding="utf-8")
# 在模型推理子进程中加载并注册自定义算子
# 在模型推理子进程中加载并注册自定义算子
load_lib_stmt = f'torch.ops.load_library(r"{LIB_PATH}")'

if load_lib_stmt not in ct_code:
    marker = "import torch_npu\n"
    if marker not in ct_code:
        marker = "import torch\n"
    if marker not in ct_code:
        raise RuntimeError("目标文件中没有找到 torch 导入语句")

    ct_code = ct_code.replace(
        marker,
        marker + load_lib_stmt + "\n",
        1,
    )
    W8A8_FILE.write_text(ct_code, encoding="utf-8")
    print("[替换] 已写入动态库加载语句")

# 准备动态库加载语句
load_lib_stmt = (
    f'torch.ops.load_library(r"{LIB_PATH}")'
)

print("[替换] 动态库路径：", LIB_PATH)
print("[替换] 目标文件读取成功")
code = W8A8_FILE.read_text(encoding='utf-8')

# 替换 npu_quant_matmul 调用为 qmm_custom 调用
old_call = (
    'x = torch_npu.npu_quant_matmul(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale,\n'
    '                                    bias=layer.bias,\n'
    '                                    output_dtype=out_dtype)'
)
new_call = (
    '# --- 替换核心: npu_quant_matmul → qmm_custom ---\n'
    '        x = torch.ops.ascendc_ops.qmm_custom(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale)\n'
    '        # --- 替换结束 ---'
)

if old_call in code:
    code = code.replace(old_call, new_call)
    W8A8_FILE.write_text(code, encoding='utf-8')
    print('[替换] 成功: npu_quant_matmul → qmm_custom')
else:
    print('[替换] 文件中未找到 npu_quant_matmul 调用，可能已替换')

# 验证替换结果
result = W8A8_FILE.read_text(encoding='utf-8')
if 'torch.ops.ascendc_ops.qmm_custom' in result:
    print('[验证] 替换已生效，当前使用 qmm_custom 算子')
else:
    print('[验证] 替换未生效，请检查文件内容')


[替换] 目标文件： /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py
[替换] 动态库路径： /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/build/libascendc_ops.so
[替换] 目标文件读取成功
[替换] 文件中未找到 npu_quant_matmul 调用，可能已替换
[验证] 替换已生效，当前使用 qmm_custom 算子


### 6.3 替换实现后执行量化推理

指定本地的量化模型权重路径，调用infer.sh执行量化推理，查看 W8A8 量化模型推理生成的文本，对比替换前的生成文本是否一致。

**注意：需要先完成第五章的量化模型推理过程，并将`W8A8_WEIGHT`指定为本地的量化模型权重的路径**

In [9]:
from pathlib import Path
import json
import os


SEARCH_ROOTS = [
    Path("/mnt/workspace"),
    Path("/mnt/data"),
    Path("/home/developer"),
    Path.home() / ".cache/modelscope/hub",
    Path.home() / ".cache/huggingface/hub",
]


def is_model_directory(path: Path) -> bool:
    """判断目录是否像一个完整的 Transformers 模型目录。"""
    if not path.is_dir():
        return False

    if not (path / "config.json").is_file():
        return False

    has_weight = (
        any(path.glob("*.safetensors"))
        or any(path.glob("pytorch_model*.bin"))
        or any(path.glob("model*.bin"))
    )

    return has_weight


def looks_like_w8a8(path: Path) -> bool:
    """
    根据目录名和 config.json 判断模型是否可能是 W8A8 量化模型。
    """
    name = path.name.lower()

    name_match = any(
        tag in name
        for tag in (
            "w8a8",
            "int8",
            "a8w8",
            "quant",
        )
    )

    config_match = False
    config_path = path / "config.json"

    try:
        config = json.loads(
            config_path.read_text(encoding="utf-8")
        )

        config_text = json.dumps(config).lower()

        config_match = any(
            tag in config_text
            for tag in (
                "compressed_tensors",
                "w8a8",
                "a8w8",
                "int8",
                "quantization_config",
            )
        )
    except Exception:
        pass

    return name_match or config_match


candidates = []

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    print(f"正在检查：{root}")

    try:
        for config_path in root.rglob("config.json"):
            model_dir = config_path.parent

            if is_model_directory(model_dir):
                if looks_like_w8a8(model_dir):
                    candidates.append(model_dir)
    except PermissionError:
        continue


# 去重
candidates = list(
    dict.fromkeys(path.resolve() for path in candidates)
)

print()
print("=" * 80)
print("发现的 W8A8/INT8 候选模型")
print("=" * 80)

if not candidates:
    print("没有发现候选模型。")
else:
    for index, path in enumerate(candidates):
        print(f"[{index}] {path}")

正在检查：/mnt/workspace
正在检查：/home/developer
正在检查：/home/developer/.cache/huggingface/hub

发现的 W8A8/INT8 候选模型
[0] /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8


In [10]:
from pathlib import Path
import importlib
import importlib.util
import os
import subprocess
import sys


# ============================================================
# 1. 工程路径
# ============================================================

TUTORIAL_DIR = Path(
    "/mnt/workspace/gitCode/cann/cann-learning-hub/"
    "tutorials/llm_inference/qwen3_8b"
).resolve()

TUTORIAL_SRC_DIR = TUTORIAL_DIR / "src"

RECIPE_ROOT = (
    TUTORIAL_SRC_DIR
    / "inference_scripts"
    / "recipe_qwen3_8b"
)

if not TUTORIAL_DIR.is_dir():
    raise FileNotFoundError(
        f"Qwen3-8B 工程目录不存在：\n{TUTORIAL_DIR}"
    )

if not TUTORIAL_SRC_DIR.is_dir():
    raise FileNotFoundError(
        f"src 目录不存在：\n{TUTORIAL_SRC_DIR}"
    )

if not RECIPE_ROOT.is_dir():
    raise FileNotFoundError(
        f"推理 recipe 目录不存在：\n{RECIPE_ROOT}"
    )

print("TUTORIAL_DIR     =", TUTORIAL_DIR)
print("TUTORIAL_SRC_DIR =", TUTORIAL_SRC_DIR)
print("RECIPE_ROOT      =", RECIPE_ROOT)


# ============================================================
# 2. 修复 Python 模块搜索路径
#
# recipe_workflow.py 内部直接导入：
#     from qwen3_npu_adaptation import ...
#
# 因此必须把 qwen3_8b/src 放入 sys.path。
# ============================================================

src_path = str(TUTORIAL_SRC_DIR)

if src_path in sys.path:
    sys.path.remove(src_path)

sys.path.insert(0, src_path)

importlib.invalidate_caches()

print("sys.path[0] =", sys.path[0])


# ============================================================
# 3. 检查 qwen3_npu_adaptation 是否可以找到
# ============================================================

adaptation_spec = importlib.util.find_spec(
    "qwen3_npu_adaptation"
)

if adaptation_spec is None:
    adaptation_candidates = list(
        TUTORIAL_DIR.rglob("qwen3_npu_adaptation.py")
    )

    candidate_text = (
        "\n".join(str(path) for path in adaptation_candidates)
        if adaptation_candidates
        else "工程中未搜索到 qwen3_npu_adaptation.py"
    )

    raise ModuleNotFoundError(
        "加入 TUTORIAL_SRC_DIR 后，仍无法导入 "
        "qwen3_npu_adaptation。\n\n"
        f"当前搜索路径首项：\n{sys.path[0]}\n\n"
        f"搜索结果：\n{candidate_text}\n\n"
        "这通常表示当前仓库文件不完整，或者使用了不匹配的分支。"
    )

print(
    "qwen3_npu_adaptation =",
    adaptation_spec.origin,
)


# ============================================================
# 4. 清除上一次错误导入留下的模块状态
# ============================================================

for module_name in list(sys.modules):
    if (
        module_name == "inference_scripts"
        or module_name.startswith("inference_scripts.")
        or module_name == "src.inference_scripts"
        or module_name.startswith("src.inference_scripts.")
    ):
        sys.modules.pop(module_name, None)

importlib.invalidate_caches()


# ============================================================
# 5. 使用原始 Notebook 的正确导入方式
# ============================================================

from inference_scripts.recipe_workflow import (
    prepare_runtime_yaml,
)

print("prepare_runtime_yaml 导入成功")


# ============================================================
# 6. 模型和 YAML 路径
# ============================================================

W8A8_WEIGHT = (
    TUTORIAL_SRC_DIR
    / "data"
    / "models"
    / "Qwen3-8B-W8A8"
)

YAML_A8W8_TEMPLATE = (
    RECIPE_ROOT
    / "models"
    / "qwen"
    / "config"
    / "qwen3_8b_a8w8_1tp.yaml"
)

YAML_A8W8_CUSTOM = (
    RECIPE_ROOT
    / "models"
    / "qwen"
    / "config"
    / "qwen3_8b_a8w8_custom.yaml"
)

if not W8A8_WEIGHT.is_dir():
    raise FileNotFoundError(
        f"量化模型权重目录不存在：\n{W8A8_WEIGHT}"
    )

if not YAML_A8W8_TEMPLATE.is_file():
    raise FileNotFoundError(
        f"模板 YAML 不存在：\n{YAML_A8W8_TEMPLATE}"
    )

print("W8A8_WEIGHT       =", W8A8_WEIGHT)
print("YAML 模板         =", YAML_A8W8_TEMPLATE)
print("YAML 输出         =", YAML_A8W8_CUSTOM)


# ============================================================
# 7. 生成运行时 YAML
# ============================================================

prepare_runtime_yaml(
    YAML_A8W8_TEMPLATE,
    YAML_A8W8_CUSTOM,
    model_path=str(W8A8_WEIGHT),
)

if not YAML_A8W8_CUSTOM.is_file():
    raise RuntimeError(
        f"运行时 YAML 没有成功生成：\n{YAML_A8W8_CUSTOM}"
    )

print("运行时 YAML 生成成功")


# ============================================================
# 8. 执行推理
# ============================================================

INFER_SCRIPT = (
    RECIPE_ROOT
    / "executor"
    / "scripts"
    / "infer.sh"
)

if not INFER_SCRIPT.is_file():
    raise FileNotFoundError(
        f"推理脚本不存在：\n{INFER_SCRIPT}"
    )

env = os.environ.copy()

python_dir = str(
    Path(sys.executable).resolve().parent
)

env["PATH"] = (
    python_dir
    + os.pathsep
    + env.get("PATH", "")
)

# infer.sh 启动的是新的子进程，需要同步设置 PYTHONPATH。
python_paths = [
    str(TUTORIAL_SRC_DIR),
    str(RECIPE_ROOT),
]

old_pythonpath = env.get("PYTHONPATH", "")

if old_pythonpath:
    python_paths.append(old_pythonpath)

env["PYTHONPATH"] = os.pathsep.join(python_paths)

print("PYTHONPATH =", env["PYTHONPATH"])
print("开始运行：", INFER_SCRIPT)

result = subprocess.run(
    [
        "bash",
        "executor/scripts/infer.sh",
        "--model",
        "qwen",
        "--yaml",
        str(YAML_A8W8_CUSTOM),
    ],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)

print("推理执行完成，返回码：", result.returncode)

TUTORIAL_DIR     = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b
TUTORIAL_SRC_DIR = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src
RECIPE_ROOT      = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b
sys.path[0] = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src
qwen3_npu_adaptation = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/qwen3_npu_adaptation.py
prepare_runtime_yaml 导入成功
W8A8_WEIGHT       = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8
YAML 模板         = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_1tp.yaml
YAML 输出         = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/infe

rm: cannot remove '/root/atc_data/': Permission denied


world_size = 1
[INFO][2026-07-22 23:21:46.469][patch_getenv.py:15] get env LOCAL_RANK = 0
[INFO][2026-07-22 23:21:46.469][patch_getenv.py:15] get env RANK_OFFSET = 0
[INFO][2026-07-22 23:21:46.470][patch_getenv.py:15] get env LC_ALL = en_US.UTF-8
[INFO][2026-07-22 23:21:46.472][patch_getenv.py:15] get env WORK_DIR = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen
[INFO][2026-07-22 23:21:46.472][patch_getenv.py:15] get env RES_PATH = res/20260722/qwen3_8b_qwen3_8b_a8w8_custom
[INFO][2026-07-22 23:21:46.472][infer.py:146] Inference Configuration
[INFO][2026-07-22 23:21:46.472][infer.py:147] InferenceConfig(model_config=ModelConfig(model_name='qwen3_8b', model_path='/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8', output_path='/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models

---
## 7 模型性能测试

### 任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程

将 QmmCustom 算子接入模型后，使用 Profiling 工具观测模型推理性能，对比自定义算子替换前后的 matmul 耗时变化。

### 7.1 准备 Profiling YAML

沿用 recipes 的 YAML + `infer.sh` 工作流，在配置中打开 Profiler，运行一次推理并收集性能数据。

通过如下操作将 YAML 配置中的enable_profiler配置为 True ：

In [11]:
from inference_scripts.recipe_workflow import prepare_profiling_yaml

YAML_A8W8_CUSTOM_PROFILER = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_custom_profiler.yaml'

prepare_profiling_yaml(YAML_A8W8_CUSTOM, YAML_A8W8_CUSTOM_PROFILER, enable_profiler=True)

{'model_config': {'model_name': 'qwen3_8b',
  'model_path': '/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8',
  'exe_mode': 'eager',
  'enable_profiler': True,
  'with_ckpt': True,
  'enable_cache_compile': False,
  'enable_static_kernel': False},
 'data_config': {'dataset': 'default', 'input_truncated_len': 4096},
 'parallel_config': {'world_size': 1,
  'attn_tp_size': 1,
  'moe_tp_size': 1,
  'embed_tp_size': 1,
  'lmhead_tp_size': 1},
 'scheduler_config': {'max_new_tokens': 256, 'batch_size': 1}}

### 7.2 启动 Profiling 推理

使用 recipes 命令启动带 Profiler 的推理，推理结束后会在结果目录中生成 `prof/` 产物。

In [12]:
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', YAML_A8W8_CUSTOM_PROFILER],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)

rm: cannot remove '/root/atc_data/': Permission denied


====================> 启动离线推理 (model=qwen, yaml=/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_custom_profiler.yaml)
world_size = 1
[INFO][2026-07-22 23:22:35.258][patch_getenv.py:15] get env LOCAL_RANK = 0
[INFO][2026-07-22 23:22:35.258][patch_getenv.py:15] get env RANK_OFFSET = 0
[INFO][2026-07-22 23:22:35.258][patch_getenv.py:15] get env LC_ALL = en_US.UTF-8
[INFO][2026-07-22 23:22:35.260][patch_getenv.py:15] get env WORK_DIR = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen
[INFO][2026-07-22 23:22:35.260][patch_getenv.py:15] get env RES_PATH = res/20260722/qwen3_8b_qwen3_8b_a8w8_custom_profiler
[INFO][2026-07-22 23:22:35.260][infer.py:146] Inference Configuration
[INFO][2026-07-22 23:22:35.260][infer.py:147] InferenceConfig(model_config=ModelConfig(model_name='qwen3_8b', model_path='/mnt/workspace/gitC

### 7.3 查看 Profiling 结果

推理完成后，定位 `prof/` 目录，重点关注其中的 `op_statistic.csv` 和 `kernel_details.csv` 这两个文件。`op_statistic.csv`中展示了模型调用的所有算子的统计信息，可快速确认自定义算子 QmmCustom 的调用情况。`kernel_details.csv`中展示了详细的接口调用信息，可自行查看筛选自己关注的算子的性能。

通过如下操作统计 `kernel_details.csv` 中的 QmmCustom 算子耗时，可自行对比自定义算子替换前后的性能差异。

In [13]:
import pandas as pd
from IPython.display import display

RES_DIR = RECIPE_ROOT / "models/qwen/res"

# 查找最新的模型 Profiling 结果
profile_roots = sorted(
    [
        p for p in RES_DIR.rglob(
            "qwen3_8b_qwen3_8b_a8w8_custom_profiler"
        )
        if p.is_dir()
    ],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
assert profile_roots, "没有找到 Profiling 结果目录"
profile_root = profile_roots[0]
print("最新 Profiling 目录：", profile_root)

def find_kernel_csv(phase):
    matches = sorted(
        (profile_root / "prof" / phase).rglob("kernel_details.csv"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    assert matches, f"没有找到 {phase} 的 kernel_details.csv"
    return matches[0]

prefill_csv = find_kernel_csv("prefill")
decode_csv = find_kernel_csv("decode")

print("Prefill CSV：", prefill_csv)
print("Decode CSV：", decode_csv)

def load_qmm_rows(csv_path):
    df = pd.read_csv(csv_path)

    mask = pd.Series(False, index=df.index)
    for column in ("Type", "Name"):
        if column in df.columns:
            mask |= df[column].astype(str).str.contains(
                r"QmmCustom|qmm_custom",
                case=False,
                na=False,
            )

    qmm = df.loc[mask].copy()
    assert not qmm.empty, f"{csv_path} 中没有找到 QmmCustom 记录"
    return qmm

def qmm_duration_stats(df, phase):
    # Profiler 有时不记录 Shape 和输出类型，先补齐，避免 groupby 产生空表
    for column in ("Input Shapes", "Output Data Types"):
        if column not in df.columns:
            df[column] = "Profiler 未记录"
        df[column] = (
            df[column]
            .fillna("Profiler 未记录")
            .replace("", "Profiler 未记录")
        )

    df["Duration(us)"] = pd.to_numeric(
        df["Duration(us)"], errors="coerce"
    )
    df = df.dropna(subset=["Duration(us)"])

    stats = (
        df.groupby(
            ["Input Shapes", "Output Data Types"],
            dropna=False,
        )["Duration(us)"]
        .agg(["mean", "min", "max", "count"])
        .reset_index()
    )

    stats.columns = [
        "Input Shapes",
        "Output Dtype",
        "Avg Duration(us)",
        "Min Duration(us)",
        "Max Duration(us)",
        "Calls",
    ]
    stats.insert(0, "Phase", phase)
    return stats

prefill_qmm = load_qmm_rows(prefill_csv)
decode_qmm = load_qmm_rows(decode_csv)

combined = pd.concat(
    [
        qmm_duration_stats(prefill_qmm, "Prefill"),
        qmm_duration_stats(decode_qmm, "Decode"),
    ],
    ignore_index=True,
)

for column in (
    "Avg Duration(us)",
    "Min Duration(us)",
    "Max Duration(us)",
):
    combined[column] = combined[column].round(3)

display(
    combined.style.set_caption(
        "QmmCustom 模型 Profiling 耗时统计"
    )
)

最新 Profiling 目录： /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260722/qwen3_8b_qwen3_8b_a8w8_custom_profiler
Prefill CSV： /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260722/qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/prefill/44b2288bb7704dbc8fd101e90bda8fa0_20807_20260722232300673_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv
Decode CSV： /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260722/qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/decode/44b2288bb7704dbc8fd101e90bda8fa0_20807_20260722232305234_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv


,Phase,Input Shapes,Output Dtype,Avg Duration(us),Min Duration(us),Max Duration(us),Calls
0,Prefill,Profiler 未记录,Profiler 未记录,62.734000,34.240000,114.522000,144
1,Decode,Profiler 未记录,Profiler 未记录,47.403000,21.121000,97.621000,432


## 8 提交要求

> 完整的提交说明参考[启航营课程实践说明](https://gitcode.com/cann/cann-launch-camp/blob/master/2026/University/HIT/First-session/README.md)。

1. 提交完整的 Jupyter Notebook（.ipynb），**保留所有单元格输出**。
2. 提交前依次运行所有单元格，确保：
   - 自定义算子编译成功；
   - 算子功能测试通过；
   - 性能数据收集成功。
3. **文件命名**：`gitcode账号_result.ipynb`。
4. **允许并且鼓励**使用 AI 工具辅助，但须独立思考并在报告中简述（解决了什么疑惑、是否引入新 bug）。
5. **不得抄袭。**

---
### 小结

本章完成了 QmmCustom 算子的开发、编译和单算子功能性能测试，并将该自定义算子接入了 Qwen3-8B 量化模型，验证功能并观测了其在模型中的性能。


### 附录
实践相关参考材料链接：
- [Ascend C 高阶api文档](https://gitcode.com/cann/asc-devkit/blob/master/docs/api/SIMD-API/%E9%AB%98%E9%98%B6API/%E9%AB%98%E9%98%B6API.md)
- [昇腾社区文档](https://www.hiascend.com/zh/document)
- [Ascend C 算子开发系列教程](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development)
- [Ascend C 算子功能调试](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development/07_Troubleshooting)
- [Matmul最佳实践样例](https://gitcode.com/cann/asc-devkit/blob/master/examples/01_simd_cpp_api/05_best_practices/01_matrix_compute/matmul_high_performance/README.md)
- [CANN内置量化矩阵乘算子实现](https://gitcode.com/cann/ops-nn/tree/master/matmul/quant_batch_matmul_v3)